## Notebook Overview

In this notebook we will build and tests several deep neural networks for day-ahead price forecasting.

We start by importing all the required python packages 

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch import nn
import torch.nn.functional as F
import torch.distributions as dist

## Experiment Settings

This block contains all the experiment settings, from the dataset configuration to the dnn parameters. 

Once you change something I suggest to re-run all the notebook cells, to confirm the usage of the new settings (you can use the "run all" button on the top of the page) 

In [2]:
args = {
    'seed': 20,
    'region': "BE",
    'predict_horizon': 24,
    'recalibration_shift_days': 200, # number of days between each re-train during the test phase
    'val_ratio': 0.2, # percentage of the train data used for validation

    'train_start': '2019-01-01',
    'train_end': '2023-10-01',
    'test_start': '2023-10-01', 
    'test_end': '2024-10-01', 

    # DNN parameters
    'target_quantiles': [i/100 for i in range(1, 100)], # this creates a vector from 0.01 to 0.99
    'perform_RevIN': True, 
    'hidden_size': 128,  
    'hidden_layers': 2,
    'activation_fn': 'ReLU', 
    'dropout_rate': 0.1,
    'mixture_components': 3,
    'context_window_days': [-1,-2,-7], # days indexes used as context before the forecast horizon
    'full_history_hours': 168, # hours used to compute RevIN stats
    'epochs': 800,
    'patience': 10,
    'learning_rate': 5e-4,
    'batch_size': 128,
    'num_workers': 4,
}

## Device And Output Folders

This block checks whether a GPU is available. If not, the notebook uses the CPU.

It also creates the folders where training logs, saved models, and final results will be stored.

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print('Using GPU')
else:
    device = torch.device('cpu')
    print('Using CPU')

# Create checkpoint and log folders
os.makedirs(f'log_dir', exist_ok=True)
os.makedirs(f'checkpoints', exist_ok=True)
os.makedirs(f'results', exist_ok=True)

Using CPU


## Loading The Data

This block loads the dataset and the parameters needed to bring predictions back to the original scale.

It also computes the final input size that will be used as input dimention by the neural network

As **past features** we just use the past values of the target (the price).

As **future features** we use the forecasts of wind, solar and load 

**Costant features** are also used to help the model learn the temporal ordering of the hours in the forecast horizon. 

In [5]:
df_raw = pd.read_csv("./BE/df_full_scaled.csv") # datasets
feature_cols = list(df_raw.columns)
feature_cols.remove('TARG__target_scaled')
feature_cols.remove('date')
df_raw['date'] = pd.to_datetime(df_raw['date'])

denorm_params = pd.read_csv("./BE/df_target_denorm_params.csv") # denormalization parameters
denorm_params['date'] = pd.to_datetime(denorm_params['date'])


cons_cols = [col for col in feature_cols if 'CONS' in col]
futu_cols = [col for col in feature_cols if 'CONS' not in col]
print("Feature columns: ", feature_cols)
print("Future columns: ", futu_cols)
print("Cons columns: ", cons_cols)

# Compute the input data shape, which is the input dimension of the DNN
# len(args['context_window_days'])*24 ---> past features, which in this case are the target values of the past days
# len(futu_cols)*args['predict_horizon'] --> future features, which are the future known values of the features
# len(cons_cols) --> static features, those are the same during the day, this is why they are not repeated for each hour
args['input_data_shape'] = len(args['context_window_days'])*24 + (len(futu_cols)*args['predict_horizon'] + len(cons_cols)) 
print("Input data shape: ", args['input_data_shape'])

Feature columns:  ['FUTU__load_f_scaled', 'FUTU__wind_f_scaled', 'FUTU__solar_f_scaled', 'CONS__wd_sin', 'CONS__wd_cos', 'CONS__ts_age']
Future columns:  ['FUTU__load_f_scaled', 'FUTU__wind_f_scaled', 'FUTU__solar_f_scaled']
Cons columns:  ['CONS__wd_sin', 'CONS__wd_cos', 'CONS__ts_age']
Input data shape:  147


## Building The Dataset

This block defines a custom PyTorch dataset. Its job is to prepare each training example in the correct format for the training, validation and test phases.

For every sample, it collects past target values, future known features, and the true future target. 

It also compute the RevIn stats (mean and std) over the history window of each feature, then we use them to normalize the past and future features. 

Mean and std are also computed over the history window of the target and are then returned in the output. We will need those values later on to denormalize the predictions. 

In [6]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_cols: list, target: str, context_window_days: list, full_history_hours: int, prediction_horizon: int, daily_aligned: bool = False):
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.target = target
        self.context_window_days = context_window_days
        self.full_history_hours = full_history_hours
        self.prediction_horizon = prediction_horizon
        self.all_features_unnorm = self.df[self.feature_cols].values.astype(np.float32)
        self.all_target_unnorm = self.df[self.target].values.astype(np.float32)
        self.dates = self.df['date'].values
        self.static_features_idx = [self.feature_cols.index(col) for col in self.feature_cols if 'CONS' in col]
        self.dynamic_features_idx = [self.feature_cols.index(col) for col in self.feature_cols if 'CONS' not in col]

        # Compute valid indices (We want our model to only predict 24 hour periods from 00:00 to 23:00) 
        total_window = self.full_history_hours + self.prediction_horizon
        all_count = max(0, len(self.df) - total_window + 1)
        if daily_aligned:
            self.valid_indices = [
                i for i in range(all_count)
                if pd.to_datetime(self.dates[i + self.full_history_hours]).hour == 0
            ]
        else:
            self.valid_indices = list(range(all_count))


    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        """
        This function is called by the DataLoader to get a sample from the dataset. This should return data in a compatible and useful format for the model.

        This function returns a tuple of 5 elements:
        - past_target_input_norm -> the past features (which are just the past values of the target) 
        - future_features_input_norm -> future features 
        - future_target_label_unnorm -> future target in original scale, it will be used to compute the loss
        - past_target_mean -> mean of the past target values
        - past_target_std -> standard deviation of the past target values

        The past target mean and std are returned because we will use them to denormalize the predictions (the inverse process of the RevIN)

        Given the idx, we can compute the past and future windows:
        - history window: [idx : idx + full_history_hours]
        - prediction horizon: [idx + full_history_hours : idx + full_history_hours + prediction_horizon]
        """
        # Map the given idx to the valid one, to start the prediction from 00:00 to 23:00 
        idx = self.valid_indices[idx]
        past_target_window_unnorm = self.all_target_unnorm[idx : idx + self.full_history_hours]
        past_features_window_unnorm = self.all_features_unnorm[idx : idx + self.full_history_hours]

        # Calculate RevIN stats
        past_target_mean = past_target_window_unnorm.mean()
        past_target_std = past_target_window_unnorm.std() + 1e-5
        past_features_mean = past_features_window_unnorm.mean(axis=0)
        past_features_std = past_features_window_unnorm.std(axis=0) + 1e-5

        # Normalize target (past and future values) 
        full_window_target_unnorm = self.all_target_unnorm[idx : idx + self.full_history_hours + self.prediction_horizon]
        full_window_target_norm = (full_window_target_unnorm - past_target_mean) / past_target_std

        # Normalize features (past and future values) 
        full_window_features_norm = (self.all_features_unnorm[idx : idx + self.full_history_hours + self.prediction_horizon] - past_features_mean) / past_features_std

        # Past Target: select specific days from the normalized history
        past_target_input_list = []
        for day_offset in self.context_window_days:
            start_in_window = self.full_history_hours + (day_offset * 24) # remember day offsets are negative! so you go back in time
            end_in_window = start_in_window + 24
            past_target_input_list.append(full_window_target_norm[start_in_window:end_in_window])


        # Future Features: prediction horizon part of full_window_features_norm
        future_features_window_norm = full_window_features_norm[self.full_history_hours : self.full_history_hours + self.prediction_horizon]


        # We flatten the past target into a single vector, and we also concatenate and flatten the future features 
        future_dynamic_features_norm = future_features_window_norm[:, self.dynamic_features_idx].flatten() 
        future_static_features_norm = future_features_window_norm[0, self.static_features_idx] # only one value for each static feature is kept, since they are constant
        future_features_input_norm = np.concatenate([future_dynamic_features_norm, future_static_features_norm])
        past_target_input_norm = np.concatenate(past_target_input_list) 

        # Future Target (is not normalized, since we will use the original scale for the loss function)
        future_target_label_unnorm = self.all_target_unnorm[idx + self.full_history_hours : idx + self.full_history_hours + self.prediction_horizon]

        return (torch.from_numpy(past_target_input_norm), 
                torch.from_numpy(future_features_input_norm), 
                torch.from_numpy(future_target_label_unnorm), 
                torch.tensor(past_target_mean, dtype=torch.float32), 
                torch.tensor(past_target_std, dtype=torch.float32))

## Evaluation Metrics

This block defines the metrics used to measure model quality.

**MAE** and **RMSE** measure point accuracy, while the others (**PICP** and **CRPS**) are for probabilistic forecasts, where the model predicts uncertainty and not only a single value.

Since PICP and CRPS are computed from quantiles, but some of the models that we use just output the distribution parameters, this block also contains utility functions to compute quantiles from a gaussian distribution and a mixture of them.

In [7]:
def MAE(pred: np.ndarray, true: np.ndarray) -> float:
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    return float(np.mean(np.abs(pred - true)))


def RMSE(pred: np.ndarray, true: np.ndarray) -> float:
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def compute_picp(y_true: np.ndarray, pred_quantiles: np.ndarray, quantiles: np.ndarray, alpha: float) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    pred_quantiles = np.asarray(pred_quantiles, dtype=np.float64)
    quantiles = np.asarray(quantiles, dtype=np.float64)

    if y_true.ndim == 1:
        y_true = y_true[:, None]
    if pred_quantiles.ndim != 3:
        raise ValueError(f"pred_quantiles must have shape (N, H, Q), got {pred_quantiles.shape}")

    target_low = (1.0 - alpha) / 2.0
    target_high = 1.0 - target_low
    idx_low = int(np.abs(quantiles - target_low).argmin())
    idx_high = int(np.abs(quantiles - target_high).argmin())

    lower = pred_quantiles[:, :, idx_low]
    upper = pred_quantiles[:, :, idx_high]
    covered = np.logical_and(lower <= y_true, y_true <= upper)
    return float(np.mean(covered))


def compute_crps_quantile(labels: np.ndarray, pred_quantiles: np.ndarray, quantiles: np.ndarray) -> float:
    labels = np.asarray(labels, dtype=np.float64)
    pred_quantiles = np.asarray(pred_quantiles, dtype=np.float64)
    quantiles = np.asarray(quantiles, dtype=np.float64)

    if pred_quantiles.ndim != 3:
        raise ValueError(f"pred_quantiles must have shape (N, H, Q), got {pred_quantiles.shape}")
    if quantiles.ndim != 1:
        raise ValueError(f"quantiles must be a 1D array, got {quantiles.shape}")

    labels_flat = labels.reshape(-1)
    pred_flat = pred_quantiles.reshape(-1, pred_quantiles.shape[-1])
    errors = labels_flat[:, None] - pred_flat
    quantiles_row = quantiles[None, :]
    loss = np.maximum(quantiles_row * errors, (quantiles_row - 1.0) * errors)
    return float(np.mean(loss))


# UTILITY FUNCTIONS

def _sample_normal_quantiles(
    loc: np.ndarray,
    scale: np.ndarray,
    quantiles: np.ndarray,
    num_samples: int = 1000,
    seed: int = 20) -> np.ndarray:
    """
    This utility function is used to sample from the normal distribution and then compute the quantiles
    """

    loc = np.asarray(loc, dtype=np.float64)
    scale = np.asarray(scale, dtype=np.float64)
    quantiles = np.asarray(quantiles, dtype=np.float64)

    rng = np.random.default_rng(seed)
    loc_tensor = torch.as_tensor(loc, dtype=torch.float64)
    scale_tensor = torch.as_tensor(scale, dtype=torch.float64)
    normal_dist = dist.Normal(loc=loc_tensor, scale=scale_tensor)

    torch.manual_seed(int(rng.integers(0, 2**31 - 1)))
    draws = normal_dist.sample((num_samples,)).cpu().numpy().T
    return np.quantile(draws, quantiles, axis=1).T


def _sample_mixnormal_quantiles(
    loc: np.ndarray,
    scale: np.ndarray,
    logits: np.ndarray,
    quantiles: np.ndarray,
    num_samples: int = 1000,
    seed: int = 20 ) -> np.ndarray:
    """
    This utility function is used to sample from the mixture of normal distribution and then compute the quantiles
    """


    loc = np.asarray(loc, dtype=np.float64)
    scale = np.asarray(scale, dtype=np.float64)
    logits = np.asarray(logits, dtype=np.float64)
    quantiles = np.asarray(quantiles, dtype=np.float64)

    if loc.ndim != 2 or scale.ndim != 2 or logits.ndim != 2:
        raise ValueError("loc, scale and logits must have shape (N, K)")
    if loc.shape != scale.shape or loc.shape != logits.shape:
        raise ValueError("loc, scale and logits must share the same shape")

    rng = np.random.default_rng(seed)
    loc_tensor = torch.as_tensor(loc, dtype=torch.float64)
    scale_tensor = torch.as_tensor(scale, dtype=torch.float64)
    logits_tensor = torch.as_tensor(logits, dtype=torch.float64)

    mix = dist.Categorical(logits=logits_tensor)
    comp = dist.Normal(loc=loc_tensor, scale=scale_tensor)
    mix_dist = dist.MixtureSameFamily(mix, comp)

    torch.manual_seed(int(rng.integers(0, 2**31 - 1)))
    draws = mix_dist.sample((num_samples,)).cpu().numpy().T
    return np.quantile(draws, quantiles, axis=1).T


def load_results_quantiles(
    model_type: str,
    quantiles: np.ndarray | None = None,
    num_samples: int = 1000,
    seed: int | None = None):
    """
    This function loads the predictions and the true values from the results folder and computes the quantiles of the predictions.
    """

    pred_path = os.path.join("results", f"{model_type}_pred.csv")
    target_path = os.path.join("results", f"{model_type}_target.csv")

    preds_df = pd.read_csv(pred_path, parse_dates=["date"]).set_index("date").sort_index()
    target_df = pd.read_csv(target_path, parse_dates=["date"]).set_index("date").sort_index()

    common_index = target_df.index.intersection(preds_df.index)
    preds_df = preds_df.loc[common_index]
    target_df = target_df.loc[common_index]
    y_true = target_df["target"].to_numpy(dtype=np.float64).reshape(-1, 1)

    if model_type == "DNN":
        quantile_columns = [col for col in preds_df.columns if col.startswith("q_")]
        quantiles = np.array([float(col.split("_", 1)[1]) for col in quantile_columns], dtype=np.float64)
        order = np.argsort(quantiles)
        quantiles = quantiles[order]
        pred_quantiles = preds_df[quantile_columns].to_numpy(dtype=np.float64)[:, order]
    elif model_type == "DNN_N":
        quantiles = np.sort(np.asarray(args["target_quantiles"] if quantiles is None else quantiles, dtype=np.float64))

        pred_quantiles = _sample_normal_quantiles(
            loc=preds_df["loc"].to_numpy(dtype=np.float64),
            scale=preds_df["scale"].to_numpy(dtype=np.float64),
            quantiles=quantiles,
            num_samples=num_samples,
            seed=args["seed"] if seed is None else seed,
        )
    elif model_type == "DNN_MIXN":
        num_components = int(args["mixture_components"])
        loc_columns = [f"loc_{i+1}" for i in range(num_components)]
        scale_columns = [f"scale_{i+1}" for i in range(num_components)]
        logit_columns = [f"logit_{i+1}" for i in range(num_components)]

        quantiles = np.sort(np.asarray(args["target_quantiles"] if quantiles is None else quantiles, dtype=np.float64))

        pred_quantiles = _sample_mixnormal_quantiles(
            loc=preds_df[loc_columns].to_numpy(dtype=np.float64),
            scale=preds_df[scale_columns].to_numpy(dtype=np.float64),
            logits=preds_df[logit_columns].to_numpy(dtype=np.float64),
            quantiles=quantiles,
            num_samples=num_samples,
            seed=args["seed"] if seed is None else seed,
        )

    return y_true, pred_quantiles[:, None, :], quantiles, common_index


def compute_results_metrics(
    model_type: str,
    alpha: float = 0.90,
    quantiles: np.ndarray | None = None,
    num_samples: int = 1000,
    seed: int | None = None,
    point_quantile: float = 0.50):
    y_true, pred_quantiles, quantiles, common_index = load_results_quantiles(
        model_type=model_type,
        quantiles=quantiles,
        num_samples=num_samples,
        seed=seed,
    )

    point_idx = int(np.abs(quantiles - point_quantile).argmin())
    point_pred = pred_quantiles[:, 0, point_idx]
    y_true_flat = y_true[:, 0]

    metrics = {
        "MAE": MAE(point_pred, y_true_flat),
        "RMSE": RMSE(point_pred, y_true_flat),
    }

    for picp_alpha in sorted({0.50, 0.90, 0.98, float(alpha)}):
        metrics[f"PICP_{int(round(picp_alpha * 100))}"] = compute_picp(
            y_true,
            pred_quantiles,
            quantiles,
            picp_alpha,
        )

    metrics["CRPS"] = compute_crps_quantile(y_true, pred_quantiles, quantiles)

    print(f"\nMetrics for {model_type} on {len(common_index)} timestamps:")
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.6f}")

    return metrics

## Loss Functions

This block defines the loss functions used during training.

The Pinball loss is used for quantile regression, while the Negative Log Likelihood is used with distributional DNN's.

In [8]:
class PinballLoss(torch.nn.Module):
    def __init__(self, quantiles: list):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, y_true, y_pred):
        # y_true: (batch, horizon)
        # y_pred: (batch, horizon, quantiles)

        if not hasattr(self, 'q_tensor') or self.q_tensor.device != y_true.device:
            self.q_tensor = torch.tensor(self.quantiles, dtype=y_true.dtype, device=y_true.device).view(1, 1, -1)

        # Broadcasting y_true to y_pred shape 
        error = y_true.unsqueeze(-1) - y_pred
        # Vectorized Pinball Loss: max(q * error, (q - 1) * error)
        loss = torch.mean(torch.maximum(self.q_tensor * error, (self.q_tensor - 1) * error))

        return loss

    def get_config(self):
        return {
            "num_quantiles": self.quantiles,
        }


class DistributionNLLLoss(torch.nn.Module):
    def forward(self, y_true, pred_dist):
        # y_true: (batch, horizon)
        # pred_dist: torch.distributions.Distribution (batch, horizon)

        # compute the probability that the true value belongs to the distribution
        # the mean is applied to compute the average over the batch and horizon
        # log is applied for numerical stability
        # negative sign is applied because we want to minimize the loss
        return -pred_dist.log_prob(y_true).mean()

## Training Loop

This block contains the main training function.

It creates the model, prepares the training and validation loaders, runs the epochs, tracks the validation loss, and saves the best version of the model.

In [9]:
def train_on_split(train_raw, suffix, feature_cols, model_class, model_type):
    best_val_loss = float('inf')
    model = model_class(args)
    model.to(device)

    # select the loss function based on the model type
    if model_type == "DNN":
        loss_fn = PinballLoss(args['target_quantiles']).to(device)
    elif model_type in {"DNN_N", "DNN_MIXN"}:
        loss_fn = DistributionNLLLoss().to(device)
    else:
        raise ValueError(f"Model type {model_type} not supported")

    optimizer = torch.optim.Adam(model.parameters(), lr=args['learning_rate'])

    run_name = f"{model_type}_{suffix}"
    checkpoint_path = os.path.join("checkpoints", f"best_model_{run_name}.pth")
    run_log_dir = os.path.join("log_dir", model_type, suffix)
    writer = SummaryWriter(log_dir=run_log_dir)
    print(f"TensorBoard logs: {run_log_dir}")

    # Build ONE dataset from all train data, with daily alignment (1 sample per day)
    full_dataset = CustomDataset(
        train_raw,
        feature_cols, 
        'TARG__target_scaled',
        context_window_days=args['context_window_days'],
        full_history_hours=args['full_history_hours'],
        prediction_horizon=args['predict_horizon'],
        daily_aligned=True  # always daily-aligned
    )

    n_samples = len(full_dataset)
    n_val = int(n_samples * args['val_ratio'])
    n_train = n_samples - n_val

    # Shuffle sample indices BEFORE splitting (in this case leads to better results) 
    all_indices = list(range(n_samples))
    np.random.shuffle(all_indices)
    train_indices = all_indices[:n_train]
    val_indices = all_indices[n_train:]

    print(f"Train/Val split: {n_train} train samples, {n_val} val samples (total: {n_samples})")

    train_subset = torch.utils.data.Subset(full_dataset, train_indices)
    val_subset = torch.utils.data.Subset(full_dataset, val_indices)

    # Build the dataloaders
    train_loader = DataLoader(
        train_subset, 
        batch_size=args['batch_size'], 
        shuffle=True, 
        num_workers=args['num_workers'], 
        pin_memory=True, 
        persistent_workers=False
    )

    val_loader = DataLoader(
        val_subset, 
        batch_size=args['batch_size'], 
        num_workers=args['num_workers'], 
        pin_memory=True, 
        persistent_workers=False
    )

    patience_counter = 0
    for epoch in range(args['epochs']): # remember: each epoch is a full pass over the train data
        # train
        model.train() # set the model to training mode
        train_losses = []
        for (past_target, future_features, future_target, target_mean, target_std) in train_loader: # each iteration is a batch of samples
            # move all the data to the device (GPU if available)
            past_target = past_target.to(device)
            future_features = future_features.to(device)
            future_target = future_target.to(device)
            target_mean = target_mean.to(device)
            target_std = target_std.to(device)

            optimizer.zero_grad() # reset the gradients

            output = model(past_target, future_features, target_mean, target_std) # forward pass

            loss = loss_fn(future_target, output) # compute the loss
            train_losses.append(loss.item())

            loss.backward() # backward pass
            optimizer.step() # update the weights

        # validation 
        model.eval() # set the model to evaluation mode
        val_losses = [] 
        with torch.no_grad(): # stop the gradient computation
            for (past_target, future_features, future_target, target_mean, target_std) in val_loader:
                past_target = past_target.to(device)
                future_features = future_features.to(device)
                future_target = future_target.to(device)
                target_mean = target_mean.to(device)
                target_std = target_std.to(device)
                output = model(past_target, future_features, target_mean, target_std) # forward pass
                loss = loss_fn(future_target, output) # compute the loss
                val_losses.append(loss.item())

        print(f"Epoch {epoch+1} - Train Loss: {np.mean(train_losses):.4f} - Val Loss: {np.mean(val_losses):.4f}")

        # Log to TensorBoard
        writer.add_scalar('Loss/train', np.mean(train_losses), epoch)
        writer.add_scalar('Loss/val', np.mean(val_losses), epoch)

        # Early stopping
        if np.mean(val_losses) < best_val_loss: 
            print(f"New best validation loss: {np.mean(val_losses):.4f}, saving model...")
            best_val_loss = np.mean(val_losses)
            torch.save(model.state_dict(), checkpoint_path)
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter > 0: print(f"Patience counter: {patience_counter} out of {args['patience']}")
        if patience_counter >= args['patience']:
            print(f"Patience reached, stopping training...")
            break

    # Clean up memory
    writer.close()
    del train_loader, val_loader, train_subset, val_subset, full_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Walk-Forward Testing

This block defines the walk-forward experiment.

We loop across all the test days, one at a time. Once a day is forecasted it is moved from ther test set to the train set. Each time we reach **recalibration_shift_days** days we train again the model from scratch using the updated train set. 

Re-training for each day would require too much time, but not performing any would leave some performance on the table. 

In [10]:
def walk_forward_experiment(model_class, model_type):
    train_raw = df_raw[(df_raw['date'] >= pd.to_datetime(args['train_start'])) & (df_raw['date'] < pd.to_datetime(args['train_end']))]
    test_raw = df_raw[(df_raw['date'] >= pd.to_datetime(args['test_start'])) & (df_raw['date'] < pd.to_datetime(args['test_end']))]

    print(f"Loaded data from {df_raw['date'].iloc[0]} to {df_raw['date'].iloc[-1]}")

    preds_list = []
    targets_list = []

    day_count = 0 
    # with each iteteration we move forward by one day
    while len(test_raw) > 0: 
        print(f"\n{'='*60}\nDay {day_count}: {test_raw['date'].iloc[0].strftime('%Y-%m-%d')}")

        # Check if we need to retrain
        if day_count % args['recalibration_shift_days'] == 0:
            run_suffix = f"week{day_count // args['recalibration_shift_days']}_{args['region']}_{args['seed']}"
            print(f"Retraining - MarketRegion: {args['region']} - Seed: {args['seed']} - Retrain split: {day_count // args['recalibration_shift_days']} / {int(365/args['recalibration_shift_days'])}")
            print("First day of train data: ", train_raw['date'].iloc[0])
            print("Last day of train data: ", train_raw['date'].iloc[-1])
            print("First day of test data: ", test_raw['date'].iloc[0])
            print("Last day of test data: ", test_raw['date'].iloc[-1])
            print(f"Train data shape: {train_raw.shape}")
            print(f"Test data shape: {test_raw.shape}")

            train_on_split(
                train_raw=train_raw,
                suffix=run_suffix,
                feature_cols=feature_cols,
                model_class=model_class,
                model_type=model_type
            )

        # Evaluation
        run_suffix = f"week{day_count // args['recalibration_shift_days']}_{args['region']}_{args['seed']}"
        checkpoint_path = os.path.join("checkpoints", f"best_model_{model_type}_{run_suffix}.pth")
        model = model_class(args) 
        model.to(device)
        model.load_state_dict(torch.load(checkpoint_path))
        model.eval()

        test_dataset = CustomDataset(
            pd.concat([train_raw, test_raw.iloc[:args['predict_horizon']]]),
            feature_cols,
            'TARG__target_scaled',
            context_window_days=args['context_window_days'],
            full_history_hours=args['full_history_hours'],
            prediction_horizon=args['predict_horizon'],
        )
        # We want the last sample of the test_dataset which corresponds to predicting the current test_raw horizon
        past_target, future_features, future_target, target_mean, target_std = test_dataset[len(test_dataset)-1]

        print("First sample of test interval: ", test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours])
        print("Last sample of test interval: ", test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours + test_dataset.prediction_horizon - 1])
        assert test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours] == test_raw['date'].iloc[0], "First sample of test interval does not match first sample of test_raw"
        assert test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours + test_dataset.prediction_horizon - 1] == test_raw['date'].iloc[args['predict_horizon'] - 1], "Last sample of test interval does not match last sample of test_raw"

        with torch.no_grad():
            past_target = past_target.unsqueeze(0).to(device)
            future_features = future_features.unsqueeze(0).to(device)
            future_target = future_target.to(device)
            target_mean = target_mean.to(device)
            target_std = target_std.to(device)
            if model_type == "DNN":
                output = model(past_target, future_features, target_mean, target_std).squeeze(0)
            elif model_type == "DNN_N":
                pred_dist = model(
                    past_target,
                    future_features,
                    target_mean,
                    target_std,
                )
                affine = pred_dist.transforms[0]
                loc = pred_dist.base_dist.loc * affine.scale + affine.loc
                scale = pred_dist.base_dist.scale * affine.scale.abs()
                output = torch.stack([loc, scale], dim=-1).squeeze(0)
            elif model_type == "DNN_MIXN":
                pred_dist = model(
                    past_target,
                    future_features,
                    target_mean,
                    target_std,
                )
                affine = pred_dist.transforms[0]
                component_dist = pred_dist.base_dist.component_distribution
                mixture_dist = pred_dist.base_dist.mixture_distribution
                loc = component_dist.loc * affine.scale.unsqueeze(-1) + affine.loc.unsqueeze(-1)
                scale = component_dist.scale * affine.scale.abs().unsqueeze(-1)
                logits = mixture_dist.logits[:, 0, :]
                horizon = loc.shape[1]
                logits = logits.unsqueeze(1).expand(-1, horizon, -1)
                output = torch.cat(
                    [loc, scale, logits],
                    dim=-1
                ).squeeze(0)
            else:
                raise ValueError(f"Model type {model_type} not supported")

        # If we have less than a full horizon left in test_raw, 
        # we should only take the entries that correspond to the test_raw entries.
        L = min(len(test_raw), args['predict_horizon'])
        if L < args['predict_horizon']:
            output = output[:L] # take the first L entries (corresponding to the start of the horizon)
            future_target = future_target[:L]

        # Denormalize if using scaled data
        test_dates = test_raw['date'].iloc[:L].values
        denorm_slice = denorm_params.set_index('date').loc[test_dates]
        p0 = torch.tensor(denorm_slice['CONS__TARG__target_trasf_p0'].values, dtype=output.dtype).to(device)
        p1 = torch.tensor(denorm_slice['CONS__TARG__target_trasf_p1'].values, dtype=output.dtype).to(device)
        if model_type == "DNN":
            output = output * p1.unsqueeze(1) + p0.unsqueeze(1)
            future_target = future_target * p1 + p0
        elif model_type == "DNN_N":
            # mu_y    = m + s * mu_z
            # sigma_y = s * sigma_z
            # output shape (L, 2) where first column is mu_y and second column is sigma_y
            output[:, 0] = output[:, 0] * p1 + p0
            output[:, 1] = output[:, 1] * p1.abs()
            future_target = future_target * p1 + p0
        elif model_type == "DNN_MIXN":
            num_components = int(args["mixture_components"])
            output[:, :num_components] = output[:, :num_components] * p1.unsqueeze(1) + p0.unsqueeze(1)
            output[:, num_components:2 * num_components] = (
                output[:, num_components:2 * num_components] * p1.abs().unsqueeze(1)
            )
            future_target = future_target * p1 + p0
        else:
            raise ValueError(f"Model type {model_type} not supported")

        output_np = output.detach().cpu().numpy()
        future_target_np = future_target.detach().cpu().numpy()
        test_dates_pd = pd.to_datetime(test_dates)

        if model_type == "DNN":
            pred_columns = [f"q_{q:.2f}" for q in args['target_quantiles']]
        elif model_type == "DNN_N":
            pred_columns = ["loc", "scale"]
        elif model_type == "DNN_MIXN":
            num_components = int(args["mixture_components"])
            pred_columns = (
                [f"loc_{i+1}" for i in range(num_components)] +
                [f"scale_{i+1}" for i in range(num_components)] +
                [f"logit_{i+1}" for i in range(num_components)]
            )
        else:
            raise ValueError(f"Model type {model_type} not supported")

        preds_list.append(pd.DataFrame(output_np, columns=pred_columns, index=test_dates_pd))
        targets_list.append(pd.DataFrame({"target": future_target_np}, index=test_dates_pd))

        # SHIFT DATASETS
        train_raw = pd.concat([train_raw, test_raw.iloc[:args['predict_horizon']]]) # concatenate the first args['predict_horizon'] days of test to train
        test_raw = test_raw.iloc[args['predict_horizon']:] # remove the first args['predict_horizon'] days of test from test
        train_raw = train_raw.iloc[args['predict_horizon']:] # remove the first args['predict_horizon'] days of train from train
        day_count += 1

    preds_df = pd.concat(preds_list).sort_index()
    target_df = pd.concat(targets_list).sort_index()
    preds_df.index.name = "date"
    target_df.index.name = "date"

    preds_path = os.path.join("results", f"{model_type}_pred.csv")
    target_path = os.path.join("results", f"{model_type}_target.csv")
    preds_df.to_csv(preds_path)
    target_df.to_csv(target_path)

    print(f"Saved predictions to {preds_path}")
    print(f"Saved targets to {target_path}")

    return preds_df, target_df

## Model 1: Quantile Regression DNN

This model predicts quantiles for each future hour, the output is the number of quantiles that we want times the number of hours to forecast. 

Quantiles describe the uncertainty for each predicted hour. 

In [11]:
# QUANTILE REGRESSION (DNN)

class DNN(nn.Module):
    def __init__(self, args):
        super(DNN, self).__init__()
        self.args = args

        self.input_layer = nn.Sequential(
            nn.Linear(self.args['input_data_shape'], self.args['hidden_size']), 
            nn.ReLU(),
            nn.Dropout(p=self.args['dropout_rate'])
        )

        self.hidden_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.args['hidden_size'], self.args['hidden_size']),
                nn.ReLU(),
                nn.Dropout(p=self.args['dropout_rate']),
            ) for _ in range(self.args['hidden_layers'] - 1)
        ])

        self.out_features = len(self.args['target_quantiles'])
        self.output_layer = nn.Linear(self.args['hidden_size'], self.out_features*self.args['predict_horizon'])

    def forward(self, past_target, future_features, target_mean, target_std): 
        # past_target: (batch, len(context_window_days)*24)
        # future_features: (batch, num_normal*24 + num_cons)

        # Inputs are already normalized by the dataset
        x = torch.cat([past_target, future_features], dim=1) # (batch, total_input_features)

        assert x.shape[1] == self.args['input_data_shape'], f"Input shape mismatch: expected {self.args['input_data_shape']}, got {x.shape[1]}"

        x = self.input_layer(x) 

        for layer in self.hidden_layers:
            x = layer(x)
        out = self.output_layer(x)

        # Reshape to (batch, horizon, quantiles)
        out = out.view(-1, self.args['predict_horizon'], self.out_features)

        # Denormalize + reshape for broadcasting
        out = out * target_std.view(-1, 1, 1) + target_mean.view(-1, 1, 1)

        # fix quantiles crossing
        out, _ = torch.sort(out, dim=-1) 
        return out 

## Running The Quantile DNN

In [12]:
walk_forward_experiment(DNN, "DNN")
compute_results_metrics(model_type="DNN")

Loaded data from 2018-12-25 00:00:00 to 2025-04-17 23:00:00

Day 0: 2023-10-01
Retraining - MarketRegion: BE - Seed: 20 - Retrain split: 0 / 1
First day of train data:  2019-01-01 00:00:00
Last day of train data:  2023-09-30 23:00:00
First day of test data:  2023-10-01 00:00:00
Last day of test data:  2024-09-30 23:00:00
Train data shape: (41616, 8)
Test data shape: (8784, 8)
TensorBoard logs: log_dir/DNN/week0_BE_20
Train/Val split: 1382 train samples, 345 val samples (total: 1727)


/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1 - Train Loss: 0.1187 - Val Loss: 0.0984
New best validation loss: 0.0984, saving model...
Epoch 2 - Train Loss: 0.0923 - Val Loss: 0.0815
New best validation loss: 0.0815, saving model...
Epoch 3 - Train Loss: 0.0816 - Val Loss: 0.0719
New best validation loss: 0.0719, saving model...
Epoch 4 - Train Loss: 0.0723 - Val Loss: 0.0664
New best validation loss: 0.0664, saving model...
Epoch 5 - Train Loss: 0.0658 - Val Loss: 0.0614
New best validation loss: 0.0614, saving model...
Epoch 6 - Train Loss: 0.0615 - Val Loss: 0.0567
New best validation loss: 0.0567, saving model...
Epoch 7 - Train Loss: 0.0579 - Val Loss: 0.0543
New best validation loss: 0.0543, saving model...
Epoch 8 - Train Loss: 0.0560 - Val Loss: 0.0528
New best validation loss: 0.0528, saving model...
Epoch 9 - Train Loss: 0.0545 - Val Loss: 0.0520
New best validation loss: 0.0520, saving model...
Epoch 10 - Train Loss: 0.0534 - Val Loss: 0.0509
New best validation loss: 0.0509, saving model...
Epoch 11 - Train Lo

/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1 - Train Loss: 0.1242 - Val Loss: 0.0969
New best validation loss: 0.0969, saving model...
Epoch 2 - Train Loss: 0.0969 - Val Loss: 0.0859
New best validation loss: 0.0859, saving model...
Epoch 3 - Train Loss: 0.0870 - Val Loss: 0.0756
New best validation loss: 0.0756, saving model...
Epoch 4 - Train Loss: 0.0769 - Val Loss: 0.0703
New best validation loss: 0.0703, saving model...
Epoch 5 - Train Loss: 0.0707 - Val Loss: 0.0667
New best validation loss: 0.0667, saving model...
Epoch 6 - Train Loss: 0.0657 - Val Loss: 0.0624
New best validation loss: 0.0624, saving model...
Epoch 7 - Train Loss: 0.0617 - Val Loss: 0.0594
New best validation loss: 0.0594, saving model...
Epoch 8 - Train Loss: 0.0588 - Val Loss: 0.0581
New best validation loss: 0.0581, saving model...
Epoch 9 - Train Loss: 0.0573 - Val Loss: 0.0572
New best validation loss: 0.0572, saving model...
Epoch 10 - Train Loss: 0.0564 - Val Loss: 0.0562
New best validation loss: 0.0562, saving model...
Epoch 11 - Train Lo

{'MAE': 13.48166312332195,
 'RMSE': 18.22717227180153,
 'PICP_50': 0.3956056466302368,
 'PICP_90': 0.8286657559198543,
 'PICP_98': 0.9578779599271403,
 'CRPS': 4.913264660409349}

## Model 2: Normal Distribution DNN

This model predicts the parameters of a normal distribution for each future hour.

In [13]:
class DNN_N(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.args = args

        self.input_layer = nn.Sequential(
            nn.Linear(self.args['input_data_shape'], self.args['hidden_size']),
            nn.ReLU(),
            nn.Dropout(p=self.args['dropout_rate'])
        )

        self.hidden_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.args['hidden_size'], self.args['hidden_size']),
                nn.ReLU(),
                nn.Dropout(p=self.args['dropout_rate']),
            ) for _ in range(self.args['hidden_layers'] - 1)
        ])

        self.out_features = 2 # loc, scale
        self.output_layer = nn.Linear(
            self.args['hidden_size'],
            self.out_features * self.args['predict_horizon']
        )

    def forward(self, past_target, future_features, target_mean, target_std):
        x = torch.cat([past_target, future_features], dim=1)

        assert x.shape[1] == self.args['input_data_shape'], (
            f"Input shape mismatch: expected {self.args['input_data_shape']}, got {x.shape[1]}"
        )

        x = self.input_layer(x)
        for layer in self.hidden_layers:
            x = layer(x)

        out = self.output_layer(x)  # (batch, 2 * horizon)

        # Match the TensorFlow implementation:
        # [loc_1, ..., loc_H, raw_scale_1, ..., raw_scale_H]
        horizon = self.args['predict_horizon']
        loc_norm = out[:, :horizon]            # (batch, horizon)
        raw_scale = out[:, horizon:]           # (batch, horizon)

        # Come in TF: 1e-3 + 3 * softplus(...)
        scale_norm = 1e-3 + 3.0 * F.softplus(raw_scale)

        target_mean = target_mean.view(-1, 1)
        target_std = target_std.view(-1, 1)

        # Distribuzione nello spazio normalizzato
        base_dist = dist.Normal(loc=loc_norm, scale=scale_norm)

        # Trasformazione affine verso lo spazio originale:
        # y = x * target_std + target_mean
        transformed_dist = dist.TransformedDistribution(
            base_dist,
            [dist.transforms.AffineTransform(loc=target_mean, scale=target_std)]
        )
        return transformed_dist

## Running The Normal Distribution DNN

In [14]:
walk_forward_experiment(DNN_N, "DNN_N")
compute_results_metrics(model_type="DNN_N")

Loaded data from 2018-12-25 00:00:00 to 2025-04-17 23:00:00

Day 0: 2023-10-01
Retraining - MarketRegion: BE - Seed: 20 - Retrain split: 0 / 1
First day of train data:  2019-01-01 00:00:00
Last day of train data:  2023-09-30 23:00:00
First day of test data:  2023-10-01 00:00:00
Last day of test data:  2024-09-30 23:00:00
Train data shape: (41616, 8)
Test data shape: (8784, 8)
TensorBoard logs: log_dir/DNN_N/week0_BE_20
Train/Val split: 1382 train samples, 345 val samples (total: 1727)


/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1 - Train Loss: 0.2122 - Val Loss: 0.1274
New best validation loss: 0.1274, saving model...
Epoch 2 - Train Loss: 0.0095 - Val Loss: -0.1103
New best validation loss: -0.1103, saving model...
Epoch 3 - Train Loss: -0.1418 - Val Loss: -0.2221
New best validation loss: -0.2221, saving model...
Epoch 4 - Train Loss: -0.2490 - Val Loss: -0.3297
New best validation loss: -0.3297, saving model...
Epoch 5 - Train Loss: -0.3402 - Val Loss: -0.4106
New best validation loss: -0.4106, saving model...
Epoch 6 - Train Loss: -0.4121 - Val Loss: -0.4691
New best validation loss: -0.4691, saving model...
Epoch 7 - Train Loss: -0.4679 - Val Loss: -0.5054
New best validation loss: -0.5054, saving model...
Epoch 8 - Train Loss: -0.5105 - Val Loss: -0.5264
New best validation loss: -0.5264, saving model...
Epoch 9 - Train Loss: -0.5375 - Val Loss: -0.5397
New best validation loss: -0.5397, saving model...
Epoch 10 - Train Loss: -0.5645 - Val Loss: -0.5557
New best validation loss: -0.5557, saving mo

/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1 - Train Loss: 0.3171 - Val Loss: 0.3563
New best validation loss: 0.3563, saving model...
Epoch 2 - Train Loss: 0.1296 - Val Loss: 0.1616
New best validation loss: 0.1616, saving model...
Epoch 3 - Train Loss: -0.0516 - Val Loss: -0.0127
New best validation loss: -0.0127, saving model...
Epoch 4 - Train Loss: -0.1831 - Val Loss: -0.1475
New best validation loss: -0.1475, saving model...
Epoch 5 - Train Loss: -0.2765 - Val Loss: -0.2358
New best validation loss: -0.2358, saving model...
Epoch 6 - Train Loss: -0.3408 - Val Loss: -0.2837
New best validation loss: -0.2837, saving model...
Epoch 7 - Train Loss: -0.3913 - Val Loss: -0.3420
New best validation loss: -0.3420, saving model...
Epoch 8 - Train Loss: -0.4257 - Val Loss: -0.3546
New best validation loss: -0.3546, saving model...
Epoch 9 - Train Loss: -0.4459 - Val Loss: -0.3806
New best validation loss: -0.3806, saving model...
Epoch 10 - Train Loss: -0.4704 - Val Loss: -0.4027
New best validation loss: -0.4027, saving mode

/tmp/ipykernel_61867/4270252456.py:69: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  loc_tensor = torch.as_tensor(loc, dtype=torch.float64)



Metrics for DNN_N on 8784 timestamps:
MAE: 13.735027
RMSE: 18.565714
PICP_50: 0.535861
PICP_90: 0.900387
PICP_98: 0.968010
CRPS: 5.003806


{'MAE': 13.73502698212427,
 'RMSE': 18.565713822939824,
 'PICP_50': 0.5358606557377049,
 'PICP_90': 0.9003870673952641,
 'PICP_98': 0.9680100182149363,
 'CRPS': 5.003806164603507}

## Model 3: Mixture Distribution DNN

This model is more flexible because it predicts a mixture of several normal distributions.

This can better represent complex forecast shapes when a single normal distribution is too simple.

In [15]:
class DNN_MIXN(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.args = args
        self.num_components = int(self.args['mixture_components'])

        self.input_layer = nn.Sequential(
            nn.Linear(self.args['input_data_shape'], self.args['hidden_size']),
            nn.ReLU(),
            nn.Dropout(p=self.args['dropout_rate'])
        )

        self.hidden_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.args['hidden_size'], self.args['hidden_size']),
                nn.ReLU(),
                nn.Dropout(p=self.args['dropout_rate']),
            ) for _ in range(self.args['hidden_layers'] - 1)
        ])

        self.out_features = 2 * self.args['predict_horizon'] * self.num_components + self.num_components
        self.output_layer = nn.Linear(self.args['hidden_size'], self.out_features)

    def forward(self, past_target, future_features, target_mean, target_std):
        x = torch.cat([past_target, future_features], dim=1)

        assert x.shape[1] == self.args['input_data_shape'], (
            f"Input shape mismatch: expected {self.args['input_data_shape']}, got {x.shape[1]}"
        )

        x = self.input_layer(x)
        for layer in self.hidden_layers:
            x = layer(x)

        out = self.output_layer(x)
        horizon = self.args['predict_horizon']
        comp_block = horizon * self.num_components

        loc_norm = out[:, :comp_block].view(-1, horizon, self.num_components)
        raw_scale = out[:, comp_block:2 * comp_block].view(-1, horizon, self.num_components)
        logits_norm = out[:, 2 * comp_block:]
        scale_norm = 1e-3 + 3.0 * F.softplus(raw_scale)
        target_mean = target_mean.view(-1, 1)
        target_std = target_std.view(-1, 1)

        logits_tiled = logits_norm.view(-1, 1, self.num_components).expand(-1, self.args['predict_horizon'], -1)

        mixture_dist = dist.Categorical(logits=logits_tiled)
        component_dist = dist.Normal(loc=loc_norm, scale=scale_norm)
        base_dist = dist.MixtureSameFamily(mixture_dist, component_dist)

        transformed_dist = dist.TransformedDistribution(
            base_dist,
            [dist.transforms.AffineTransform(loc=target_mean, scale=target_std)]
        )
        return transformed_dist

## Running The Mixture Distribution DNN

In [ ]:
walk_forward_experiment(DNN_MIXN, "DNN_MIXN")
compute_results_metrics(model_type="DNN_MIXN")

Loaded data from 2018-12-25 00:00:00 to 2025-04-17 23:00:00

Day 0: 2023-10-01
Retraining - MarketRegion: BE - Seed: 20 - Retrain split: 0 / 1
First day of train data:  2019-01-01 00:00:00
Last day of train data:  2023-09-30 23:00:00
First day of test data:  2023-10-01 00:00:00
Last day of test data:  2024-09-30 23:00:00
Train data shape: (41616, 8)
Test data shape: (8784, 8)
TensorBoard logs: log_dir/DNN_MIXN/week0_BE_20
Train/Val split: 1382 train samples, 345 val samples (total: 1727)


/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1 - Train Loss: 0.2388 - Val Loss: 0.1210
New best validation loss: 0.1210, saving model...
Epoch 2 - Train Loss: 0.0429 - Val Loss: -0.1600
New best validation loss: -0.1600, saving model...
Epoch 3 - Train Loss: -0.1578 - Val Loss: -0.2850
New best validation loss: -0.2850, saving model...
Epoch 4 - Train Loss: -0.2792 - Val Loss: -0.4021
New best validation loss: -0.4021, saving model...
Epoch 5 - Train Loss: -0.3891 - Val Loss: -0.4769
New best validation loss: -0.4769, saving model...
Epoch 6 - Train Loss: -0.4652 - Val Loss: -0.5578
New best validation loss: -0.5578, saving model...
Epoch 7 - Train Loss: -0.5197 - Val Loss: -0.6029
New best validation loss: -0.6029, saving model...
Epoch 8 - Train Loss: -0.5649 - Val Loss: -0.6276
New best validation loss: -0.6276, saving model...
Epoch 9 - Train Loss: -0.5893 - Val Loss: -0.6448
New best validation loss: -0.6448, saving model...
Epoch 10 - Train Loss: -0.6140 - Val Loss: -0.6681
New best validation loss: -0.6681, saving mo

/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1 - Train Loss: 0.3381 - Val Loss: 0.3097
New best validation loss: 0.3097, saving model...
Epoch 2 - Train Loss: 0.1303 - Val Loss: 0.0259
New best validation loss: 0.0259, saving model...
Epoch 3 - Train Loss: -0.0829 - Val Loss: -0.1007
New best validation loss: -0.1007, saving model...
Epoch 4 - Train Loss: -0.1992 - Val Loss: -0.2226
New best validation loss: -0.2226, saving model...
Epoch 5 - Train Loss: -0.2910 - Val Loss: -0.3074
New best validation loss: -0.3074, saving model...
Epoch 6 - Train Loss: -0.3724 - Val Loss: -0.3703
New best validation loss: -0.3703, saving model...
Epoch 7 - Train Loss: -0.4128 - Val Loss: -0.4040
New best validation loss: -0.4040, saving model...
Epoch 8 - Train Loss: -0.4556 - Val Loss: -0.4298
New best validation loss: -0.4298, saving model...
Epoch 9 - Train Loss: -0.4801 - Val Loss: -0.4456
New best validation loss: -0.4456, saving model...
Epoch 10 - Train Loss: -0.5061 - Val Loss: -0.4778
New best validation loss: -0.4778, saving mode

{'MAE': 13.61666357171397,
 'RMSE': 18.40898356639078,
 'PICP_50': 0.45412112932604737,
 'PICP_90': 0.8827413479052824,
 'PICP_98': 0.9701730418943534,
 'CRPS': 4.932280426236613}

## TensorBoard View

In [ ]:
import socket
import subprocess
import sys
import time
from IPython.display import HTML, display

def _find_free_port(start_port=6006):
    port = start_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) != 0:
                return port
        port += 1

if "_tensorboard_process" in globals() and _tensorboard_process.poll() is None:
    port = _tensorboard_port
else:
    port = _find_free_port(6006)
    cmd = [
        sys.executable,
        "-m",
        "tensorboard.main",
        "--logdir",
        "log_dir",
        "--host",
        "127.0.0.1",
        "--port",
        str(port),
        "--reload_interval",
        "5",
    ]
    _tensorboard_process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    _tensorboard_port = port
    time.sleep(3)

url = f"http://127.0.0.1:{port}/"
print(f"TensorBoard running at: {url}")
display(HTML(f'<p><a href="{url}" target="_blank">Open TensorBoard in a new tab</a></p>'))
display(HTML(f'<iframe src="{url}" width="100%" height="900"></iframe>'))

TensorBoard running at: http://127.0.0.1:6006/


/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
